In [ ]:
async def show_results():
    async with AsyncExitStack() as stack:
        accounts_server = await stack.enter_async_context(
            MCPServerStdio(trader_mcp_params[0], client_session_timeout_seconds=30)
        )
        for name, info in TRADERS.items():
            balance = await accounts_server.call_tool("get_balance", {"name": name})
            holdings = await accounts_server.call_tool("get_holdings", {"name": name})
            print(f"\n{'='*40}")
            print(f"{name} ({info['style']})")
            print(f"{'='*40}")
            print(f"Cash Balance: {balance}")
            print(f"Holdings: {holdings}")

await show_results()

## Check Results\n\nRead both accounts back from the MCP server to see what each trader bought.

In [ ]:
results = await asyncio.gather(
    *[run_trader(name, info["strategy"]) for name, info in TRADERS.items()]
)

## Run Both Traders\n\nBoth traders run concurrently with `asyncio.gather`, just like the stock trading floor in Lab 5.

In [ ]:
MAX_TURNS = 30

async def run_trader(name: str, strategy: str):
    async with AsyncExitStack() as stack:
        trader_servers = [
            await stack.enter_async_context(
                MCPServerStdio(params, client_session_timeout_seconds=120)
            )
            for params in trader_mcp_params
        ]
        researcher_servers = [
            await stack.enter_async_context(
                MCPServerStdio(params, client_session_timeout_seconds=120)
            )
            for params in researcher_mcp_params
        ]

        researcher = Agent(
            name="CryptoResearcher",
            instructions=researcher_instructions(),
            model="gpt-4o-mini",
            mcp_servers=researcher_servers,
        )
        researcher_tool = researcher.as_tool(
            tool_name="Researcher",
            tool_description="Researches crypto news, trends, and on-chain data based on your request.",
        )

        # Read current account state via the accounts MCP server
        account_data = await trader_servers[0].call_tool("get_balance", {"name": name})
        holdings_data = await trader_servers[0].call_tool("get_holdings", {"name": name})
        account_summary = f"Balance: {account_data}, Holdings: {holdings_data}"

        trader_agent = Agent(
            name=name,
            instructions=trader_instructions(name),
            model="gpt-4o-mini",
            tools=[researcher_tool],
            mcp_servers=trader_servers,
        )

        message = trade_message(name, strategy, account_summary)
        print(f"\n{'='*60}")
        print(f"Running {name}...")
        print(f"{'='*60}")

        result = await Runner.run(trader_agent, message, max_turns=MAX_TURNS)
        print(f"\n{name}'s summary: {result.final_output}")
        return result.final_output

## Build and Run Traders\n\nEach trader gets:\n- A **researcher sub-agent** (with Fetch + Brave Search MCP servers) exposed as a tool\n- Direct access to **crypto market** and **accounts** MCP servers\n\nThis mirrors the agent-as-tool composition pattern from Lab 4.

In [ ]:
async def reset_accounts():
    async with AsyncExitStack() as stack:
        accounts_server = await stack.enter_async_context(
            MCPServerStdio(trader_mcp_params[0], client_session_timeout_seconds=30)
        )
        for name, info in TRADERS.items():
            result = await accounts_server.call_tool(
                "reset_account", {"name": name, "strategy": info["strategy"]}
            )
            print(f"{name} ({info['style']}): {result}")

await reset_accounts()

## Reset Accounts\n\nInitialize both traders with $10,000 and their strategies. Uses the `reset_account` tool on the accounts MCP server.

In [ ]:
def researcher_instructions():
    return (
        "You are a crypto market researcher. Search the web for crypto news, "
        "on-chain developments, regulatory updates, and market sentiment. "
        "Summarize findings with actionable trading insights. "
        "If the web search tool hits a rate limit, fall back to fetching web pages directly. "
        f"Current datetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    )


def trader_instructions(name):
    return (
        f"You are {name}, a cryptocurrency trader. Your account is under the name {name}. "
        f"You have tools to look up crypto prices, get top cryptos by market cap, "
        f"buy and sell crypto (fractional amounts allowed), and a researcher for news. "
        f"Use these tools to research, decide, and execute trades. "
        f"After trading, reply with a 2-3 sentence summary of what you did and why."
    )


def trade_message(name, strategy, account):
    return (
        f"Look for new crypto trading opportunities based on your strategy.\n"
        f"Use the researcher to find relevant news first, then check prices and execute trades.\n\n"
        f"Your strategy:\n{strategy}\n\n"
        f"Your current account:\n{account}\n\n"
        f"Current datetime: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Your account name is {name}."
    )

## Agent Prompts

In [ ]:
TRADERS = {
    "Satoshi": {
        "style": "HODLer",
        "strategy": (
            "You are Satoshi, a conviction-driven long-term crypto investor. "
            "You believe in the fundamental value of decentralized networks. "
            "You focus on BTC and ETH as core positions, allocating smaller amounts to "
            "established L1s (SOL, ADA, AVAX) you believe have strong tech and adoption. "
            "You buy on dips and almost never sell. You ignore short-term volatility "
            "and think in multi-year time horizons. You do deep research on protocol "
            "fundamentals before buying anything."
        ),
    },
    "Vitalik": {
        "style": "Swing Trader",
        "strategy": (
            "You are Vitalik, an active momentum-based crypto trader. "
            "You watch for breakout patterns, volume spikes, and trending narratives. "
            "You trade a wider range of tokens including mid-caps (LINK, UNI, ARB, OP, SUI). "
            "You take quick profits when momentum fades and cut losses fast. "
            "You use the top cryptos list and price tools frequently to spot momentum. "
            "You research trending crypto news to find narrative-driven trades. "
            "You aim to outperform a simple buy-and-hold through active management."
        ),
    },
}

## Trader Personalities\n\nTwo traders with opposing crypto philosophies, each starting with $10,000.

In [ ]:
SERVER_DIR = os.path.dirname(os.path.abspath("__file__"))

trader_mcp_params = [
    {"command": "uv", "args": ["run", os.path.join(SERVER_DIR, "crypto_accounts_server.py")]},
    {"command": "uv", "args": ["run", os.path.join(SERVER_DIR, "crypto_market_server.py")]},
]

brave_api_key = os.getenv("BRAVE_API_KEY", "")

researcher_mcp_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "env": {"BRAVE_API_KEY": brave_api_key},
    },
]

## MCP Server Config\n\nTwo custom MCP servers (in this folder):\n- `crypto_market_server.py` — wraps CoinGecko for price lookups and top-coins listing\n- `crypto_accounts_server.py` — manages trader accounts with fractional crypto holdings in SQLite\n\nPlus public MCP servers for the researcher: Fetch (web scraping) and Brave Search (needs `BRAVE_API_KEY` in `.env`).

In [ ]:
import os, asyncio
from datetime import datetime
from contextlib import AsyncExitStack
from dotenv import load_dotenv
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

# Crypto Trading Floor\n\nAdapts the stock trading floor from Labs 4-5 to the cryptocurrency market.\n\nKey differences from the stock version:\n- **CoinGecko API** (free, no API key) instead of Polygon.io\n- **Fractional holdings** — buy 0.01 BTC, not just whole shares\n- **24/7 market** — crypto never closes, no market-hours check needed\n- **Two trader personalities**: a long-term HODLer vs a momentum Swing Trader\n\nUses the same MCP patterns taught in the course: custom MCP servers for market data and accounts, the OpenAI Agents SDK for orchestration, and a researcher sub-agent for web research.